# Task 1: Particle Field Evolution (GNO)

This notebook is intentionally self-contained and manual.
It trains a true GNO model on `../output/particle_dataset.npz` and runs sanity plots.


In [ ]:
from pathlib import Path
import json, time, numpy as np, matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Robust base-path resolution.
CWD = Path.cwd().resolve()
if (CWD / 'final-2' / 'output').exists():
    BASE = CWD / 'final-2'
elif (CWD.name == 'notebooks') and (CWD.parent / 'output').exists():
    BASE = CWD.parent
else:
    BASE = CWD

DATA_PATH = BASE / 'output' / 'particle_dataset.npz'
OUT_DIR = BASE / 'output' / 'task1_training'
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Base:', BASE)
print('DATA_PATH:', DATA_PATH)
print('Exists?:', DATA_PATH.exists())
print('Device:', DEVICE)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH}. Run preprocess first.')


In [ ]:
ds = np.load(DATA_PATH, allow_pickle=True)
X = ds['inputs_particle_norm'].astype(np.float32)
X_raw = ds['inputs_particle'].astype(np.float32)
Y = ds['targets_particle_norm'].astype(np.float32)
feature_names = [str(x) for x in ds['feature_names'].tolist()]
target_names = [str(x) for x in ds['target_names'].tolist()]
frame_ranges = list(ds['frame_ranges'])
train_frames = ds['train_frames'].astype(np.int64)
val_frames = ds['val_frames'].astype(np.int64)
test_frames = ds['test_frames'].astype(np.int64)
print('X', X.shape, 'Y', Y.shape, 'frames', len(frame_ranges))
print('features:', feature_names)
print('targets:', target_names)


In [ ]:
# Quick sanity scatter (pick a frame with many particles, not necessarily frame 0)
frame_sizes = np.array([int(r[3]) - int(r[2]) for r in frame_ranges], dtype=np.int64)
big_idx = int(np.argmax(frame_sizes))
case, fr, s, e = frame_ranges[big_idx]
s, e = int(s), int(e)
xyz = X_raw[s:e, :3]  # raw coordinates are easier to read

print(f'Plotting frame index={big_idx}, case={case}, frame={fr}, n_particles={e-s}')

plt.figure(figsize=(6,5))
plt.scatter(xyz[:,0], xyz[:,2], s=1, alpha=0.35)
plt.title(f'Sample frame cloud (raw): {case} / {fr}')
plt.xlabel('x')
plt.ylabel('z')
plt.tight_layout()
plt.show()


In [ ]:
class FrameDataset(Dataset):
    def __init__(self, X, Y, frame_ranges, frame_ids):
        self.X, self.Y = X, Y
        self.frame_ranges = frame_ranges
        self.frame_ids = [int(i) for i in frame_ids]
    def __len__(self):
        return len(self.frame_ids)
    def __getitem__(self, i):
        fi = self.frame_ids[i]
        _, _, s, e = self.frame_ranges[fi]
        s, e = int(s), int(e)
        return torch.from_numpy(self.X[s:e]), torch.from_numpy(self.Y[s:e])

def collate_fn(batch):
    xs, ys = zip(*batch)
    return list(xs), list(ys)

train_loader = DataLoader(FrameDataset(X,Y,frame_ranges,train_frames), batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(FrameDataset(X,Y,frame_ranges,val_frames), batch_size=1, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(FrameDataset(X,Y,frame_ranges,test_frames), batch_size=1, shuffle=False, collate_fn=collate_fn)


In [ ]:
import inspect

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as exc:
    raise RuntimeError('Need neuralop with GNOBlock for this notebook.') from exc

def rel_l2(pred, tgt, eps=1e-12):
    d = (pred - tgt).reshape(pred.shape[0], -1)
    t = tgt.reshape(tgt.shape[0], -1)
    return (torch.linalg.norm(d, dim=1) / torch.linalg.norm(t, dim=1).clamp_min(eps)).mean()

class ParticleGNO(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=64, n_layers=2, radius=0.12):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Linear(hidden, hidden))

        sig = inspect.signature(GNOBlock.__init__)
        p = sig.parameters
        extra = {}
        if 'use_torch_scatter_reduce' in p:
            extra['use_torch_scatter_reduce'] = False
        if 'use_open3d_neighbor_search' in p:
            extra['use_open3d_neighbor_search'] = False

        self.blocks = nn.ModuleList([
            GNOBlock(
                in_channels=hidden, out_channels=hidden, coord_dim=3, radius=radius,
                transform_type='linear', reduction='mean', pos_embedding_type='transformer',
                pos_embedding_channels=8, channel_mlp_layers=[hidden, hidden, hidden],
                **extra,
            ) for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, out_dim))

    def forward(self, x):
        pos = x[:, :3]
        h = self.enc(x)
        for blk, norm in zip(self.blocks, self.norms):
            u = blk(y=pos, x=pos, f_y=h)
            if u.ndim == 3 and u.shape[0] == 1:
                u = u.squeeze(0)
            h = norm(h + u)
        return self.head(h)

model = ParticleGNO(X.shape[1], Y.shape[1]).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
# Optimizer (NeuralOperator Adam/AdamW when available)
opt_cls = None
try:
    import neuralop.training as nt
    opt_cls = getattr(nt, 'AdamW', None) or getattr(nt, 'Adam', None)
except Exception:
    pass
if opt_cls is None:
    opt_cls = torch.optim.AdamW

opt = opt_cls(model.parameters(), lr=1e-3, weight_decay=1e-6)
sch = torch.optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.5)


In [ ]:
# Faster training settings for large particle datasets.
EPOCHS = 80
MAX_NODES = 1024
FRAMES_PER_EPOCH = 64
PRINT_EVERY_BATCH = 8

hist = []
best = np.inf
best_state = None

# Keep references to split datasets for frame-level sampling.
train_frame_dataset = train_loader.dataset
val_frame_dataset = val_loader.dataset
test_frame_dataset = test_loader.dataset

def eval_loader(loader, max_eval_frames=40):
    model.eval()
    rels, mses = [], []
    with torch.no_grad():
        for bidx, (xs, ys) in enumerate(loader):
            if bidx >= max_eval_frames:
                break
            x, y = xs[0], ys[0]
            if x.shape[0] > MAX_NODES:
                idx = torch.randperm(x.shape[0])[:MAX_NODES]
                x, y = x[idx], y[idx]
            x, y = x.to(DEVICE), y.to(DEVICE)
            p = model(x)
            rels.append(float(rel_l2(p.unsqueeze(0), y.unsqueeze(0)).item()))
            mses.append(float(torch.mean((p-y)**2).item()))
    return float(np.mean(rels)), float(np.mean(mses))

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    model.train()
    losses = []

    n_train_frames = len(train_frame_dataset)
    use_n = min(FRAMES_PER_EPOCH, n_train_frames)
    frame_pick = np.random.choice(n_train_frames, size=use_n, replace=False)

    for step, fi in enumerate(frame_pick, start=1):
        x, y = train_frame_dataset[int(fi)]
        if x.shape[0] > MAX_NODES:
            idx = torch.randperm(x.shape[0])[:MAX_NODES]
            x, y = x[idx], y[idx]

        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        p = model(x)
        loss = rel_l2(p.unsqueeze(0), y.unsqueeze(0))
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))

        if step % PRINT_EVERY_BATCH == 0:
            print(f'[ep {ep:03d}] step {step:03d}/{use_n} loss={loss.item():.6f}', flush=True)

    sch.step()

    tr = float(np.mean(losses))
    vr, vm = eval_loader(val_loader, max_eval_frames=24)
    elapsed = time.time() - t0

    hist.append({'epoch':ep, 'train_loss':tr, 'val_rel_l2':vr, 'val_mse':vm, 'sec':elapsed})

    if vr < best:
        best = vr
        best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}

    print(f'[epoch {ep:03d}] train={tr:.6f} val_rel={vr:.6f} val_mse={vm:.6f} time={elapsed/60:.2f} min', flush=True)

if best_state is not None:
    model.load_state_dict(best_state)
tr, tm = eval_loader(test_loader, max_eval_frames=32)
print('TEST rel_l2=', tr, 'mse=', tm)


In [ ]:
torch.save(model.state_dict(), OUT_DIR / 'best_particle_gno_model.pt')
(OUT_DIR / 'history.json').write_text(json.dumps(hist, indent=2))

plt.figure(figsize=(7,4))
plt.plot([h['epoch'] for h in hist], [h['train_loss'] for h in hist], label='train')
plt.plot([h['epoch'] for h in hist], [h['val_rel_l2'] for h in hist], label='val_rel_l2')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()
print('saved to', OUT_DIR)
